# Lab 3 - Image Captioning (Task 3.2)

Image captioning network on Flickr8k. CNN encoder (ResNet50) + LSTM decoder, with a switchable fusion step (concat / add / mult / diff) so we can compare them across the group.

The notebook follows the steps from the assignment - each section below has its own code block:

| Section | Code |
|---|---|
| 1. Image Feature Extraction | `EncoderCNN` - frozen ResNet50 + projection |
| 2. Sequence Model for Language Processing | `TextEncoder` - embedding + LSTM, returns hidden states |
| 3. Combining Image and Text Data | `Combine` - image projection + fusion (concat/add/mult/diff) + output FC |
| 4. Training the Network | training loop |
| 5. Generating Captions | greedy decode + sample plot + wandb table |
| 6. Evaluation | BLEU-1 / BLEU-4 |

Setup / config / vocab / dataset cells come first, before the numbered sections.

## Setup

In [ ]:
!pip install -q nltk wandb
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

### Config

Each group member runs one of: `concat`, `add`, `mult`, `diff`.

In [ ]:
fusion = 'concat'

dataset_dir = '.'
image_dir = dataset_dir + '/Images'
caption_file = dataset_dir + '/captions.txt'

embed_dim = 256
hidden_dim = 256
encoder_dim = 2048  # resnet50 output

batch_size = 32
num_epochs = 15
lr = 1e-3
max_len = 40
freq_thresh = 5

seed = 42

In [ ]:
import os
import re
import random
import numpy as np
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import wandb

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
wandb.init(
    project='D7047E-Lab3',
    name='captioning-' + fusion,
    config={
        'fusion': fusion,
        'embed_dim': embed_dim,
        'hidden_dim': hidden_dim,
        'encoder_dim': encoder_dim,
        'lr': lr,
        'batch_size': batch_size,
        'epochs': num_epochs,
        'max_len': max_len,
        'freq_thresh': freq_thresh,
        'encoder': 'resnet50',
        'seed': seed,
    },
)

### Vocabulary

Build word2idx from captions.txt. Words seen fewer than `freq_thresh` times collapse to UNK.
Specials: PAD=0, START=1, END=2, UNK=3.

In [ ]:
PAD, START, END, UNK = 0, 1, 2, 3

def tokenize(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

class Vocabulary:
    def __init__(self, freq_threshold):
        self.freq_threshold = freq_threshold
        self.word2idx = {'<PAD>': PAD, '<START>': START, '<END>': END, '<UNK>': UNK}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build(self, captions):
        counter = Counter()
        for c in captions:
            counter.update(tokenize(c))
        i = len(self.word2idx)
        for word, cnt in counter.items():
            if cnt >= self.freq_threshold:
                self.word2idx[word] = i
                self.idx2word[i] = word
                i += 1
        print('vocab size:', len(self.word2idx))

    def encode(self, caption):
        ids = [self.word2idx.get(w, UNK) for w in tokenize(caption)]
        return [START] + ids + [END]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == END:
                break
            if i in (PAD, START):
                continue
            out.append(self.idx2word.get(i, '<UNK>'))
        return ' '.join(out)

    def __len__(self):
        return len(self.word2idx)

In [ ]:
# load captions.txt (header + 'image_name,caption' per line)
all_captions = []
all_data = []

with open(caption_file, 'r') as f:
    next(f)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(',', 1)
        if len(parts) != 2:
            continue
        img, cap = parts[0].strip(), parts[1].strip()
        all_data.append((img, cap))
        all_captions.append(cap)

print(len(all_data), 'pairs,', len(set(d[0] for d in all_data)), 'unique images')
print('example:', all_data[0])

vocab = Vocabulary(freq_thresh)
vocab.build(all_captions)
wandb.config.update({'vocab_size': len(vocab)})

### Dataset / DataLoader

Splitting by image (not by pair) so the same image doesn't end up in both train and test.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FlickrDataset(Dataset):
    def __init__(self, data, image_dir, vocab, transform, max_len):
        self.data = data
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        name, caption = self.data[idx]
        img = Image.open(os.path.join(self.image_dir, name)).convert('RGB')
        img = self.transform(img)

        ids = self.vocab.encode(caption)
        if len(ids) < self.max_len:
            ids = ids + [PAD] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len - 1] + [END]
        return img, torch.tensor(ids, dtype=torch.long)

unique_imgs = list({d[0] for d in all_data})
random.shuffle(unique_imgs)
cut = int(0.9 * len(unique_imgs))
train_set = set(unique_imgs[:cut])
test_set = set(unique_imgs[cut:])

train_data = [(n, c) for n, c in all_data if n in train_set]
test_data = [(n, c) for n, c in all_data if n in test_set]
print('train:', len(train_data), ' test:', len(test_data))

train_ds = FlickrDataset(train_data, image_dir, vocab, transform, max_len)
test_ds = FlickrDataset(test_data, image_dir, vocab, transform, max_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

## 1. Image Feature Extraction

Pretrained ResNet50 with the classification head removed. The 2048-dim avgpool feature is projected down to `embed_dim` by a trainable FC + BN. Backbone weights are frozen, so only the projection layer learns.

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_dim, encoder_dim=2048):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # drop the final fc, keep up to avgpool
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        for p in self.resnet.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(encoder_dim, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, images):
        with torch.no_grad():
            x = self.resnet(images)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.bn(self.fc(x)))
        return x

encoder = EncoderCNN(embed_dim, encoder_dim).to(device)
print('trainable encoder params:', sum(p.numel() for p in encoder.parameters() if p.requires_grad))

## 2. Sequence Model for Language Processing

Word embedding -> single-layer LSTM. Takes the caption tokens (teacher forcing during training) and returns the per-step hidden states, which we'll combine with the image vector in the next section.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, batch_first=True)

    def forward(self, captions):
        e = self.embedding(captions)         # (B, T, embed_dim)
        out, _ = self.lstm(e)                # (B, T, hidden_dim)
        return out

text_model = TextEncoder(len(vocab), embed_dim, hidden_dim).to(device)
print('text model params:', sum(p.numel() for p in text_model.parameters()))

## 3. Combining Image and Text Data

Merge architecture (Tanti & Gatt) - image embedding and the LSTM hidden states are kept independent and combined per timestep, then a single FC predicts the next word.

Fusion options:
- `concat`: stack along last dim, then linear over `embed_dim + hidden_dim`
- `add` / `mult` / `diff`: project the LSTM output to `embed_dim` first, then elementwise op, then linear over `embed_dim`

In [ ]:
class Combine(nn.Module):
    def __init__(self, embed_dim, hidden_dim, encoder_dim, vocab_size, fusion='concat'):
        super().__init__()
        self.fusion = fusion

        # image side projection (ResNet feat -> embed_dim)
        self.img_fc = nn.Linear(encoder_dim, embed_dim)
        self.img_bn = nn.BatchNorm1d(embed_dim)

        if fusion == 'concat':
            self.out_fc = nn.Linear(embed_dim + hidden_dim, vocab_size)
        else:
            # need same dim for elementwise ops
            self.txt_proj = nn.Linear(hidden_dim, embed_dim)
            self.out_fc = nn.Linear(embed_dim, vocab_size)

        self.dropout = nn.Dropout(0.3)

    def fuse(self, img, txt):
        if self.fusion == 'concat':
            return torch.cat([img, txt], dim=-1)
        txt = self.txt_proj(txt)
        if self.fusion == 'add':
            return img + txt
        if self.fusion == 'mult':
            return img * txt
        if self.fusion == 'diff':
            return img - txt
        raise ValueError(self.fusion)

    def forward(self, image_features, lstm_out):
        img = torch.relu(self.img_bn(self.img_fc(image_features)))   # (B, embed_dim)
        img = img.unsqueeze(1).expand(-1, lstm_out.size(1), -1)      # broadcast over time
        fused = self.dropout(self.fuse(img, lstm_out))
        return self.out_fc(fused)                                    # (B, T, V)

combine = Combine(embed_dim, hidden_dim, embed_dim, len(vocab), fusion=fusion).to(device)

n_params = sum(p.numel() for p in text_model.parameters()) + sum(p.numel() for p in combine.parameters())
print('fusion:', fusion, ' decoder + combine params:', n_params)
wandb.config.update({'model_params': n_params})

## 4. Training the Network

Cross-entropy with `ignore_index=PAD`. Teacher forcing: input is `caption[:-1]`, target is `caption[1:]`. We train the encoder's projection layer + the text encoder + the combine block (the ResNet backbone stays frozen). Gradient clipping at 5.0 to keep things stable.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
params = (
    list(encoder.fc.parameters())
    + list(encoder.bn.parameters())
    + list(text_model.parameters())
    + list(combine.parameters())
)
optimizer = optim.Adam(params, lr=lr)

wandb.watch(combine, log='gradients', log_freq=200)

for epoch in range(num_epochs):
    encoder.train()
    text_model.train()
    combine.train()

    running_loss = 0.0
    n = 0

    for batch_idx, (images, captions) in enumerate(train_loader):
        images = images.to(device)
        captions = captions.to(device)

        x = captions[:, :-1]
        y = captions[:, 1:]

        feats = encoder(images)
        lstm_out = text_model(x)
        logits = combine(feats, lstm_out)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
        optimizer.step()

        running_loss += loss.item()
        n += 1
        if batch_idx % 50 == 0:
            wandb.log({'batch_loss': loss.item()})

    avg = running_loss / n
    wandb.log({'epoch': epoch + 1, 'train_loss': avg})
    print(f'epoch {epoch+1}/{num_epochs}  loss={avg:.4f}')

print('done.')

## 5. Generating Captions

Greedy decoding - feed the predicted token back in until we hit END or `max_len`. Then plot a handful of test images with their generated and reference captions, and push them to a wandb table for easier browsing.

In [ ]:
@torch.no_grad()
def generate_caption(image, max_len=25):
    encoder.eval()
    text_model.eval()
    combine.eval()
    image = image.unsqueeze(0).to(device)
    feats = encoder(image)

    seq = [START]
    for _ in range(max_len):
        inp = torch.tensor([seq], dtype=torch.long, device=device)
        lstm_out = text_model(inp)
        logits = combine(feats, lstm_out)
        nxt = logits[0, -1].argmax().item()
        seq.append(nxt)
        if nxt == END:
            break
    return vocab.decode(seq)

@torch.no_grad()
def caption_from_path(path):
    img = Image.open(path).convert('RGB')
    return generate_caption(transform(img))

In [ ]:
test_imgs = list({n for n, _ in test_data})
random.shuffle(test_imgs)
samples = test_imgs[:10]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, name in enumerate(samples):
    path = os.path.join(image_dir, name)
    img = Image.open(path).convert('RGB')
    pred = caption_from_path(path)
    refs = [c for n, c in all_data if n == name]
    ref = refs[0] if refs else ''

    axes[i].imshow(img)
    axes[i].set_title(f'pred: {pred}\nref:  {ref[:60]}', fontsize=8)
    axes[i].axis('off')

plt.suptitle('fusion = ' + fusion)
plt.tight_layout()
plt.savefig('sample_captions.png', dpi=150, bbox_inches='tight')
plt.show()

wandb.log({'sample_captions_plot': wandb.Image('sample_captions.png')})

In [ ]:
# wandb table - same samples, easier to scroll through in the dashboard
table = wandb.Table(columns=['image', 'generated', 'reference', 'name'])

for name in samples:
    path = os.path.join(image_dir, name)
    pred = caption_from_path(path)
    refs = [c for n, c in all_data if n == name]
    ref = refs[0] if refs else ''
    table.add_data(wandb.Image(path, caption=pred), pred, ref, name)

wandb.log({'caption_samples': table})

## 6. Evaluation

Optional per the assignment, but corpus BLEU-1 / BLEU-4 across the test set against all available reference captions per image. Smoothing method 1 (avoids zero scores when higher n-grams have no overlap).

In [ ]:
ref_dict = {}
for name, cap in test_data:
    ref_dict.setdefault(name, []).append(tokenize(cap))

references = []
hypotheses = []

for i, name in enumerate(ref_dict):
    path = os.path.join(image_dir, name)
    if not os.path.exists(path):
        continue
    pred = caption_from_path(path).split()
    references.append(ref_dict[name])
    hypotheses.append(pred)
    if (i + 1) % 100 == 0:
        print(i + 1, '/', len(ref_dict))

smooth = SmoothingFunction().method1
bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth)
bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

print('BLEU-1:', round(bleu1, 4))
print('BLEU-4:', round(bleu4, 4))
wandb.log({'bleu1': bleu1, 'bleu4': bleu4})

### Save + finish wandb

In [ ]:
ckpt = {
    'encoder_state_dict': encoder.state_dict(),
    'text_model_state_dict': text_model.state_dict(),
    'combine_state_dict': combine.state_dict(),
    'vocab': vocab,
    'config': {
        'fusion': fusion,
        'embed_dim': embed_dim,
        'hidden_dim': hidden_dim,
        'vocab_size': len(vocab),
    },
}
torch.save(ckpt, f'captioning_{fusion}.pth')
print('saved captioning_' + fusion + '.pth')

wandb.finish()